In [1]:
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("JAX_PLATFORMS", "cpu")

def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "develop_doc").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import braincell
from braincell.io import compare_swc_with_neuron

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)


/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.1 is installed, but it is not compatible with the installed jaxlib version 0.6.2, so it will not be used.
  warnings.warn(


braincell version: 0.0.8
repo_root: /home/swl/braincell


# `neuron_diff`: scan cached NeuroMorpho folders and compare with `NEURON`

This notebook reads the existing cache under `develop_doc/data/neuromorpho/<neuron_id>/`, finds the standardized `SWC`, and prints the `braincell` vs `neuron` report from `braincell.io.compare_swc_with_neuron(...)`.


In [2]:
def load_cached_metadata(folder):
    return json.loads((Path(folder) / "metadata.json").read_text(encoding="utf-8"))


def repo_path(path_value):
    path = Path(path_value)
    return path if path.is_absolute() else repo_root / path


def find_standard_swc(folder, metadata):
    for item in metadata.get("download_items", []):
        if item.get("kind") == "standard":
            path = repo_path(item["path"])
            if path.exists():
                return path

    swc_paths = sorted(Path(folder).glob("*.swc"))
    return swc_paths[0] if swc_paths else None


In [3]:
output_root = repo_root / "develop_doc" / "data" / "neuromorpho"
neuron_metric_names = None

if output_root.exists():
    cache_dirs = sorted(path for path in output_root.iterdir() if path.is_dir())
else:
    cache_dirs = []

neuron_runs = []
summary = {
    "scanned": 0,
    "reported": 0,
    "skipped": 0,
    "errors": 0,
}

if not cache_dirs:
    print("No cached neuron folders were found under", output_root)
else:
    for folder in cache_dirs:
        summary["scanned"] += 1
        metadata_path = folder / "metadata.json"
        if not metadata_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: missing metadata.json")
            continue

        try:
            metadata = load_cached_metadata(folder)
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to read metadata.json: {exc}")
            continue

        swc_path = find_standard_swc(folder, metadata)
        if swc_path is None or not swc_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no standardized SWC found")
            continue

        try:
            neuron_result = compare_swc_with_neuron(swc_path, metric_names=neuron_metric_names)
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to build neuron report for {swc_path}: {exc}")
            continue

        summary["reported"] += 1
        neuron_runs.append(
            {
                "folder": str(folder),
                "neuron_id": metadata.get("neuron_id"),
                "neuron_name": metadata.get("neuron_name"),
                "swc_path": str(swc_path),
                "report": neuron_result,
            }
        )

        print("=" * 80)
        print(f"neuron_id: {metadata.get('neuron_id')}")
        print(f"neuron_name: {metadata.get('neuron_name')}")
        print(f"folder: {folder}")
        print(f"swc_path: {swc_path}")
        print()

        for metric_name in neuron_result["selected_metrics"]:
            braincell_record = neuron_result["braincell"][metric_name]
            neuron_record = neuron_result["neuron"][metric_name]
            diff_record = neuron_result["diff"][metric_name]

            if braincell_record["available"]:
                braincell_text = f"{braincell_record['value']:.6f}"
                if braincell_record["unit"]:
                    braincell_text += f" {braincell_record['unit']}"
            else:
                braincell_text = f"n/a ({braincell_record['reason']})"

            if neuron_record["available"]:
                neuron_text = f"{neuron_record['value']:.6f}"
                if neuron_record["unit"]:
                    neuron_text += f" {neuron_record['unit']}"
            else:
                neuron_text = f"n/a ({neuron_record['reason']})"

            if diff_record["available"]:
                rel_text = "n/a" if diff_record["rel_diff"] is None else f"{diff_record['rel_diff']:.6%}"
                diff_text = f"abs_diff={diff_record['abs_diff']:.6f}, rel_diff={rel_text}"
            else:
                diff_text = f"unavailable ({diff_record['reason']})"

            print(f"{metric_name}:")
            print(f"  braincell: {braincell_text}")
            print(f"  neuron: {neuron_text}")
            print(f"  diff: {diff_text}")

    print("=" * 80)
    print("neuron report summary")
    print("scanned:", summary["scanned"])
    print("reported:", summary["reported"])
    print("skipped:", summary["skipped"])
    print("errors:", summary["errors"])

neuron_runs


neuron_id: 10047
neuron_name: TypeA-10
folder: /home/swl/braincell/develop_doc/data/neuromorpho/10047
swc_path: /home/swl/braincell/develop_doc/data/neuromorpho/10047/TypeA-10.CNG.swc

total_length:
  braincell: 1402.989300 um
  neuron: 1402.989048 um
  diff: abs_diff=0.000252, rel_diff=0.000018%
total_area:
  braincell: 607.072550 um^2
  neuron: 607.072464 um^2
  diff: abs_diff=0.000086, rel_diff=0.000014%
total_volume:
  braincell: 208.996479 um^3
  neuron: 208.996484 um^3
  diff: abs_diff=0.000005, rel_diff=0.000003%
mean_radius:
  braincell: 0.068866 um
  neuron: 0.068866 um
  diff: abs_diff=0.000000, rel_diff=0.000004%
n_branches:
  braincell: 22.000000 count
  neuron: 22.000000 count
  diff: abs_diff=0.000000, rel_diff=0.000000%
n_stems:
  braincell: 1.000000 count
  neuron: 1.000000 count
  diff: abs_diff=0.000000, rel_diff=0.000000%
n_bifurcations:
  braincell: 10.000000 count
  neuron: 10.000000 count
  diff: abs_diff=0.000000, rel_diff=0.000000%
max_branch_order:
  braincell:

--No graphics will be displayed.


neuron_id: 10048
neuron_name: TypeB-06
folder: /home/swl/braincell/develop_doc/data/neuromorpho/10048
swc_path: /home/swl/braincell/develop_doc/data/neuromorpho/10048/TypeB-06.CNG.swc

total_length:
  braincell: 2578.078735 um
  neuron: 2578.078600 um
  diff: abs_diff=0.000135, rel_diff=0.000005%
total_area:
  braincell: 1337.034576 um^2
  neuron: 1337.034514 um^2
  diff: abs_diff=0.000063, rel_diff=0.000005%
total_volume:
  braincell: 1459.301667 um^3
  neuron: 1459.301604 um^3
  diff: abs_diff=0.000063, rel_diff=0.000004%
mean_radius:
  braincell: 0.082540 um
  neuron: 0.082540 um
  diff: abs_diff=0.000000, rel_diff=0.000001%
n_branches:
  braincell: 76.000000 count
  neuron: 76.000000 count
  diff: abs_diff=0.000000, rel_diff=0.000000%
n_stems:
  braincell: 3.000000 count
  neuron: 3.000000 count
  diff: abs_diff=0.000000, rel_diff=0.000000%
n_bifurcations:
  braincell: 37.000000 count
  neuron: 37.000000 count
  diff: abs_diff=0.000000, rel_diff=0.000000%
max_branch_order:
  brainc

[{'folder': '/home/swl/braincell/develop_doc/data/neuromorpho/10047',
  'neuron_id': 10047,
  'neuron_name': 'TypeA-10',
  'swc_path': '/home/swl/braincell/develop_doc/data/neuromorpho/10047/TypeA-10.CNG.swc',
  'report': {'path': '/home/swl/braincell/develop_doc/data/neuromorpho/10047/TypeA-10.CNG.swc',
   'selected_metrics': ('total_length',
    'total_area',
    'total_volume',
    'mean_radius',
    'n_branches',
    'n_stems',
    'n_bifurcations',
    'max_branch_order',
    'max_path_distance'),
   'braincell': {'total_length': {'available': True,
     'value': 1402.9893003716925,
     'unit': 'um'},
    'total_area': {'available': True,
     'value': 607.0725504872064,
     'unit': 'um^2'},
    'total_volume': {'available': True,
     'value': 208.99647904898885,
     'unit': 'um^3'},
    'mean_radius': {'available': True,
     'value': 0.06886623953571565,
     'unit': 'um'},
    'n_branches': {'available': True, 'value': 22, 'unit': 'count'},
    'n_stems': {'available': True